# Rollout Cost Lab: first real GPU test

Run this notebook on **Google Colab (GPU runtime)** or **Kaggle (T4 GPU, Internet enabled)**. Use Run all for setup and a six-episode smoke test. The pilot and A/A trials are disabled by default.

The notebook uses one GPU, Qwen2.5-Coder-3B-Instruct, FP16, and a 4,096-token context. Packages for the runner and vLLM live in separate environments from the notebook kernel. Results are development measurements; no real-GPU compatibility or solve rate has been established yet.

Keep the notebook interactive. vLLM is used locally by the benchmark. Download results before ending the session. If a cell fails, run the final download cell to save its diagnostics.


In [1]:
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

IS_KAGGLE = Path("/kaggle/working").is_dir()
BASE = Path("/kaggle/working") if IS_KAGGLE else Path("/content")
if not BASE.is_dir():
    raise RuntimeError("Open this notebook in a Colab or Kaggle GPU runtime.")

ROOT = BASE / "rollout-cost"
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/arnabdas1999/GPU-Cost-Optimization.git", str(ROOT)], check=True)
if not (ROOT / "src/rollout_cost/notebook.py").exists():
    raise RuntimeError("This checkout predates notebook support. Update it with git pull --ff-only, then rerun.")

os.chdir(ROOT)
os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.insert(0, str(ROOT / "src"))
from rollout_cost.notebook import check_gpu, run_logged, latest_run, check_result, archive_results

check_gpu(ROOT)
print("Repository:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())


GPU 0: Tesla T4; VRAM: 15360 MiB; compute capability: 7.5; driver: 580.82.07
Kernel Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Available CPUs: [0, 1]

Repository: 1c5cac6d23e782b02bc2a2728b8f6b538e4cf08a


## Session settings

The exact vLLM release is configurable in case the assigned driver's CUDA support requires a different release. Setup checks CUDA and doctor before measuring anything. Start with the defaults below. A 3B model may need prompt/budget calibration after reviewing its transcripts.

Leave the longer steps off for the first Run all. After a successful smoke test, set RUN_PILOT to True and run the pilot cell. A/A consists of four full runs.


In [2]:
VLLM_VERSION = "0.31.0"
MODEL_REPO = "Qwen/Qwen2.5-Coder-3B-Instruct"
RUN_PILOT = False
RUN_AA = False
SESSION_TAG = globals().get("SESSION_TAG") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SMOKE_LABEL = "notebook-smoke-" + SESSION_TAG
PILOT_LABEL = "notebook-pilot-" + SESSION_TAG


## Install, pin the model, validate fixtures, and check vLLM

This downloads the model and installs vLLM under /tmp, without replacing the kernel's PyTorch. Setup starts and stops vLLM for doctor. Logs are saved to notebook-setup.log. If installation or CUDA startup fails, keep the output and download the archive for diagnosis.


In [ ]:
try:
    run_logged(ROOT, ["bash", "scripts/notebook-setup.sh"], "notebook-setup.log",
               {"VLLM_VERSION": VLLM_VERSION, "MODEL_REPO": MODEL_REPO})
finally:
    archive_results(ROOT)

import shutil
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv was not found after setup; inspect notebook-setup.log.")
RC = [UV, "run", "--extra", "dashboard", "--locked", "rollout-cost"]


Running: bash scripts/notebook-setup.sh

== GPU
Tesla T4, 15360 MiB, 580.82.07
CPUs available: 2

== uv
uv 0.13.4 (x86_64-unknown-linux-gnu)

== project dependencies
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 19 packages in 41ms
Prepared 17 packages in 1.56s
Installed 17 packages in 38ms
 + anyio==4.15.1
 + certifi==2026.10.23
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 + iniconfig==2.3.1
 + packaging==26.3
 + pluggy==1.7.0
 + prometheus-client==0.26.0
 + psycopg==3.3.6
 + psycopg-binary==3.3.6
 + pygments==2.21.0
 + pytest==9.1.1
 + pyyaml==6.0.3
 + rollout-cost-lab==0.1.0 (from file:///content/rollout-cost)
 + typing-extensions==4.16.0

== vLLM and model
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/rollout-cost-vllm-0.31.0
Activate with: source /tmp/rollout-cost-vllm-0.31.0/bin/activate
Using Python 3.12.3 environment at: /tmp/rollout-cost-vllm-0.31.0


## Six-episode GPU smoke test

The runner starts and stops its own vLLM server. Success requires a real completed run, no infrastructure errors, at least one solved task, protocol errors below 20%, no wall timeouts, and verified server drain. Read the report and two transcripts before longer tests.

If this quality check fails, the artifacts still contain useful GPU results. A zero solve rate needs calibration; it does not by itself identify an installation fault.


In [ ]:
from IPython.display import Markdown, display

try:
    run_logged(ROOT, RC + ["run", "--config", "configs/notebook-smoke.pinned.yaml",
                          "--split", "dev", "--label", SMOKE_LABEL], "notebook-smoke.log")
    SMOKE_DIR = latest_run(ROOT, SMOKE_LABEL)
    display(Markdown((SMOKE_DIR / "report.md").read_text()))
    check_result(SMOKE_DIR)
    print("Transcripts:", SMOKE_DIR / "transcripts")
finally:
    archive_results(ROOT)


## Optional 24-episode pilot

Enable RUN_PILOT in the settings cell after the smoke check succeeds. Keep the same model, pinned revision, tool budget and context settings across comparisons. Review the pilot and calibrate once before freezing settings.


In [ ]:
if RUN_PILOT:
    check_result(latest_run(ROOT, SMOKE_LABEL))
    try:
        run_logged(ROOT, RC + ["run", "--config", "configs/notebook.pinned.yaml",
                              "--split", "dev", "--label", PILOT_LABEL], "notebook-pilot.log")
        PILOT_DIR = latest_run(ROOT, PILOT_LABEL)
        display(Markdown((PILOT_DIR / "report.md").read_text()))
        check_result(PILOT_DIR)
    finally:
        archive_results(ROOT)
else:
    print("Pilot skipped. Enable RUN_PILOT after reviewing the smoke result.")


## Optional A/A noise calibration

Enable RUN_AA only after a successful pilot. The expected verdict is A/A CALIBRATION. The held-out split is empty, so this uses dev. Further sweep and paired-validation commands are in docs/NOTEBOOK_RUNBOOK.md.


In [ ]:
if RUN_AA:
    check_result(latest_run(ROOT, PILOT_LABEL))
    try:
        run_logged(ROOT, RC + ["paired", "--baseline", "configs/notebook.pinned.yaml",
                              "--candidate", "configs/notebook.pinned.yaml", "--split", "dev",
                              "--trials", "2", "--name", "notebook-aa-" + SESSION_TAG], "notebook-aa.log")
    finally:
        archive_results(ROOT)
else:
    print("A/A skipped for this session.")


## Download results, then stop the GPU session

You can run this cell even after setup or a test fails. The archive contains run directories, configs, logs and package versions. Model downloads, package environments and environment-variable secrets are excluded.

On Kaggle, download notebook-results.tgz from the working/output file browser if the link is unavailable. Once the download is saved, end the GPU session using the platform's runtime/session controls. Stopping vLLM alone does not end your GPU allocation.


In [ ]:
ARCHIVE = archive_results(ROOT)
if IS_KAGGLE:
    from IPython.display import FileLink
    display(FileLink(str(ARCHIVE)))
else:
    from google.colab import files
    files.download(str(ARCHIVE))
